In [2]:
import numpy as np
import os
from tqdm import tqdm
import pandas as pd

In [5]:
#path = '/home/kale-chen/Documents/CASToR/castor_v3.2/config/scanner/TPPT.lut'
#path = '/home/kale-chen/Documents/PET/TimeCalibration/LineSourceStudy/remapping/hori_geomasked_timecut.bin'
#path = '/home/kale-chen/Documents/CASToR/images/center_geomasked_remap_tof.cdf'
#path = '/home/kale-chen/Documents/PET/TPPTvis/data_analysis/old_source_hw_trigger_on_1800sec_test_vertical_17_horizontal_23_v2_coinc_toplors.bin'
#path#= '/home/kale-chen/Documents/PET/MDA_03162026/Data/hcenter_pcut.bin'
#path = '/home/kale-chen/Documents/PET/MDA_03162026/Data/hcenter_pcut_mask.bin'
#path = '/home/kale-chen/Documents/PET/TimeCalibration/MiniPET/80long_pcut.bin'
#path = '/home/kale-chen/Documents/PET/MDA_04112026/Data/run3_tcut_cwc_cal.bin'
#path = '/home/kale-chen/Documents/PET/TPPTvis/scanner/FullFlatScanner.lut'
#path = '/home/kale-chen/Documents/PET/MDA_10112025/4mmtpc.bin'
#path = '/home/kale-chen/Documents/PET/Energy Normalization/event_counts_norm.bin'
#path = '/home/kale-chen/Documents/CASToR/images/1723a.cdf'
#path = '/home/kale-chen/Documents/PET/TimeCalibration/LineSourceStudy/1_points_run2_HighInt_5min_coinc_pcut.bin'
#path = '/home/kale-chen/Documents/PET/MiniPET/GAGG/LYSO_pcut.bin'
#path = '/home/kale-chen/Documents/PET/Spatial Resolution/Data2/2532.bin'
path = '/home/kale-chen/Documents/PET/TPPT2026/data/pr0.bin'
num_cols = 4
data_type = np.int16

num_rows = os.path.getsize(path) // (num_cols * data_type().itemsize)
data = np.memmap(path, dtype = data_type, mode = 'r', shape = (num_rows, num_cols))
#data = data[:,0]
print(os.path.getsize(path))
print(data.shape)
print(num_rows)
print(data[0:10])
print(data[-6:-1])
#print(np.min(data[:, 0]), np.max(data[:, 0]))
#print(np.min(data[:, 1]), np.max(data[:, 1]))

17757480
(2219685, 4)
2219685
[[  354   719   841     5]
 [ 2221  2431  5164     5]
 [  300   867  5775     5]
 [  898  2014   855     5]
 [  806   566    10     5]
 [ 2606  2899  1015     5]
 [   85   533   557     5]
 [   31  2022  4772     5]
 [ 1355  1372   397     5]
 [  462  1012 -1252     5]]
[[1798 1662 4423   54]
 [ 685  886 -375   54]
 [1468 2038 -164   54]
 [2386 2192  313   54]
 [1084 1535 5469   54]]


In [6]:
size = os.path.getsize(path)
print(size, size / 305135102)

4882161632 16.0


In [3]:
print(np.min(data[:, 0]))
print(np.max(data[:, 0]))
print(np.min(data[:, 1]))
print(np.max(data[:, 1]))
print(np.min(data[:, 2]))
print(np.max(data[:, 2]))


-75
4699
0
127
-117
4108


In [21]:
geomoffsets = np.fromfile('/home/kale-chen/Documents/PET/TimeCalibration/LineSourceStudy/remapping/hailmaryfullgeo.bin', dtype = np.int32).reshape(3072, 3072)
print(geomoffsets[0:5, 0:5])

[[    0  -388     0     0     0]
 [  -61     0     0  -454     0]
 [    0  -317     0  -706  9175]
 [    0     0     0     0 -4602]
 [-5956 -5056     0     0     0]]


In [10]:
# Cutter and saver
out_data_file = os.path.join(os.path.dirname(path), 'run2coinc_pcut_pes.bin')
if os.path.exists(out_data_file):
    os.remove(out_data_file)
for i in tqdm(range(0, num_rows, 1000000), desc='Event counts'):
    chunk = data[i:i+1000000]
    chunk = chunk[chunk[:,3] > 200]
    with open(out_data_file, 'ab') as f:
        chunk.tofile(f)


Event counts: 100%|██████████| 45/45 [00:00<00:00, 483.29it/s]


In [15]:
mask = (data[:, 2] > -500) & (data[:, 2] < 500)
data = data[mask]
print(data.shape)
data.tofile('/home/kale-chen/Documents/PET/MDA_04112026/Data/run3_tcut_cwc_cal_500ps.bin')

(1174269, 3)


In [ ]:
mask = ((data[:, 2] > 500) & (data[:, 2] < 2000)) | ((data[:, 2] < -500) & (data[:, 2] > -2000))
maskright = ((data[:, 2] > 500) & (data[:, 2] < 2000))
maskleft = ((data[:, 2] < -500) & (data[:, 2] > -2000))
dataall = data[mask]
dataright = data[maskright]
dataleft = data[maskleft]
print(dataall.shape, dataright.shape, dataleft.shape)
dataall.tofile('/home/kale-chen/Documents/PET/MDA_10112025/run1_pcut_start100_cal_outside500ps.bin')
#dataright.tofile('/home/kale-chen/Documents/PET/MDA_10112025/run1_pcut_start100_cal_outside500ps_right.bin')
#dataleft.tofile('/home/kale-chen/Documents/PET/MDA_10112025/run1_pcut_start100_cal_outside500ps_left.bin')



(87781, 3) (41294, 3) (46487, 3)


In [13]:
zeros = 0
for i in range(len(data) - 1):
    if data[i, 0] == data[i + 1, 0] or data[i, 1] == data[i + 1, 1]:
        zeros += 1
print(zeros)


1392


In [ ]:
df = pd.DataFrame(data)
df.to_csv('/home/kale-chen/Documents/PET/TPPTvis/scanner/TPPT_Scanner_map_vis.csv', index=False, header=False)

In [16]:
test_value = 1
array = np.array([test_value], dtype = np.int32)
array.tofile('test.bin')
testread = np.fromfile('test.bin', dtype = np.float32)
print(testread)

[1.e-45]


In [5]:
# For cutting a binary file
path = '/home/kale-chen/Documents/PET/TimeCalibration/ToUpload/linepcut.bin'
num_cols = 4
data_type = np.int16
out_data_file = os.path.join(os.path.dirname(path), 'linepcut3col.bin')
num_rows = os.path.getsize(path) // (num_cols * data_type().itemsize)
data = np.memmap(path, dtype = data_type, mode = 'r', shape = (num_rows, num_cols))
for i in tqdm(range(0, num_rows, 1000000)):
    chunk = data[i:i+1000000]
    chunk = chunk[:, [0, 1, 2]]
    with open(out_data_file, 'ab') as f:
        chunk.tofile(f)
print(os.path.getsize(path), os.path.getsize(out_data_file))

100%|██████████| 196/196 [00:13<00:00, 14.77it/s]

1561634288 1171225716
